# 02b DFSI Numeric Debugging

This notebook debugs the DFSI reproduction after correcting the equation to the requested log-space form with no +1 on population. It does not start sensitivity analysis and does not impute the 11 incomplete rows.

## Setup

The notebook locates the repository root by searching upward for the expected raw CSV files, then writes debugging outputs to `outputs/tables`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError("Could not find the repository root containing data/raw with the expected CSV files.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "dfsi": RAW_DIR / "DFSI.csv",
    "flooded_area": RAW_DIR / "District_FloodedArea.csv",
    "flood_impact": RAW_DIR / "District_FloodImpact.csv",
    "flood_inventory": RAW_DIR / "India_Flood_Inventory_v3.csv",
}

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness


## Load Raw Sources

Raw CSV values are loaded as supplied.

In [2]:
dfsi = pd.read_csv(PATHS["dfsi"], low_memory=False)
flooded_area = pd.read_csv(PATHS["flooded_area"], low_memory=False)
flood_impact = pd.read_csv(PATHS["flood_impact"], low_memory=False)
flood_inventory = pd.read_csv(PATHS["flood_inventory"], low_memory=False)

for name, table in {
    "DFSI.csv": dfsi,
    "District_FloodedArea.csv": flooded_area,
    "District_FloodImpact.csv": flood_impact,
    "India_Flood_Inventory_v3.csv": flood_inventory,
}.items():
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]:,} columns")

DFSI.csv: 744 rows x 3 columns
District_FloodedArea.csv: 732 rows x 4 columns
District_FloodImpact.csv: 732 rows x 5 columns
India_Flood_Inventory_v3.csv: 6,876 rows x 23 columns


## Exact Component Source Columns

This table records the exact source column or derived count used for each component.

In [3]:
component_sources = pd.DataFrame([
    {
        "component": "flood_events",
        "source_dataset": "India_Flood_Inventory_v3.csv",
        "exact_source_column_used": "UEI counted after exploding Districts; District_LGD_Codes retained for audit",
        "working_column": "flood_events",
        "numeric_conversion": "pd.to_numeric(flood_events, errors='coerce').astype('float64')",
        "note": "Unmatched parsed inventory counts are treated as zero events for the district-level count component.",
    },
    {
        "component": "mean_flood_duration",
        "source_dataset": "District_FloodImpact.csv",
        "exact_source_column_used": "Mean_Flood_Duration",
        "working_column": "component_mean_flood_duration",
        "numeric_conversion": "pd.to_numeric(Mean_Flood_Duration, errors='coerce').astype('float64')",
        "note": "Missing values are preserved as missing.",
    },
    {
        "component": "fatalities",
        "source_dataset": "District_FloodImpact.csv",
        "exact_source_column_used": "Human_fatality",
        "working_column": "component_fatalities",
        "numeric_conversion": "pd.to_numeric(Human_fatality, errors='coerce').astype('float64')",
        "note": "Used exactly as supplied after numeric conversion.",
    },
    {
        "component": "injured",
        "source_dataset": "District_FloodImpact.csv",
        "exact_source_column_used": "Human_injured",
        "working_column": "component_injured",
        "numeric_conversion": "pd.to_numeric(Human_injured, errors='coerce').astype('float64')",
        "note": "Used exactly as supplied after numeric conversion.",
    },
    {
        "component": "flooded_area",
        "source_dataset": "District_FloodedArea.csv",
        "exact_source_column_used": "Corrected_Percent_Flooded_Area",
        "working_column": "component_flooded_area",
        "numeric_conversion": "pd.to_numeric(Corrected_Percent_Flooded_Area, errors='coerce').astype('float64')",
        "note": "Corrected field is used; original flooded-area and permanent-water columns are retained in the comparison output.",
    },
    {
        "component": "population",
        "source_dataset": "District_FloodImpact.csv",
        "exact_source_column_used": "Population",
        "working_column": "component_population",
        "numeric_conversion": "pd.to_numeric(Population, errors='coerce').astype('float64')",
        "note": "The specified equation for this debugging pass uses log10(Population), with no +1 applied.",
    },
])

display(component_sources)
component_sources.to_csv(TABLE_DIR / "dfsi_component_source_columns.csv", index=False)

,component,source_dataset,exact_source_column_used,working_column,numeric_conversion,note
0,flood_events,India_Flood_Inventory_v3.csv,UEI counted after exploding Districts; District_LGD_Codes retained for audit,flood_events,"pd.to_numeric(flood_events, errors='coerce').astype('float64')",Unmatched parsed inventory counts are treated as zero events for the district-level count component.
1,mean_flood_duration,District_FloodImpact.csv,Mean_Flood_Duration,component_mean_flood_duration,"pd.to_numeric(Mean_Flood_Duration, errors='coerce').astype('float64')",Missing values are preserved as missing.
2,fatalities,District_FloodImpact.csv,Human_fatality,component_fatalities,"pd.to_numeric(Human_fatality, errors='coerce').astype('float64')",Used exactly as supplied after numeric conversion.
3,injured,District_FloodImpact.csv,Human_injured,component_injured,"pd.to_numeric(Human_injured, errors='coerce').astype('float64')",Used exactly as supplied after numeric conversion.
4,flooded_area,District_FloodedArea.csv,Corrected_Percent_Flooded_Area,component_flooded_area,"pd.to_numeric(Corrected_Percent_Flooded_Area, errors='coerce').astype('float64')",Corrected field is used; original flooded-area and permanent-water columns are retained in the comparison output.
5,population,District_FloodImpact.csv,Population,component_population,"pd.to_numeric(Population, errors='coerce').astype('float64')","The specified equation for this debugging pass uses log10(Population), with no +1 applied."


## Helpers

In [4]:
def make_review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def split_list_cell(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",")]


def clean_lgd_code(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.casefold() in {"none", "nan", "na", "<na>"}:
        return pd.NA
    return text

## Parse Flood Events

Flood events are counted as distinct `UEI` values after exploding the inventory district list. LGD district codes are retained for audit, but the final join cannot be LGD-based because the district summary sources do not include LGD identifiers.

In [5]:
event_rows = []
event_parse_audit_records = []

for _, row in flood_inventory.reset_index(names="source_row_id").iterrows():
    districts = split_list_cell(row["Districts"])
    district_codes = split_list_cell(row["District_LGD_Codes"])

    if not districts:
        event_parse_audit_records.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "issue_type": "missing_district_list",
            "district_count": 0,
            "district_lgd_code_count": len(district_codes),
            "State": row.get("State"),
            "State_Codes": row.get("State_Codes"),
        })
        continue

    if district_codes and len(districts) != len(district_codes):
        event_parse_audit_records.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "issue_type": "district_name_lgd_code_count_mismatch",
            "district_count": len(districts),
            "district_lgd_code_count": len(district_codes),
            "State": row.get("State"),
            "State_Codes": row.get("State_Codes"),
        })

    for position, district_name in enumerate(districts):
        lgd_code = district_codes[position] if position < len(district_codes) else pd.NA
        event_rows.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "district_position": position + 1,
            "district_name_from_inventory": district_name,
            "district_review_key": make_review_key(district_name),
            "district_lgd_code": clean_lgd_code(lgd_code),
            "event_state_text": row.get("State"),
            "event_state_codes_text": row.get("State_Codes"),
        })

event_districts = pd.DataFrame(event_rows)
event_parse_audit = pd.DataFrame(event_parse_audit_records)

event_counts_by_name = (
    event_districts.groupby("district_review_key", dropna=False)
    .agg(
        flood_events=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_codes_seen=("district_lgd_code", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_code_count=("district_lgd_code", lambda s: int(s.dropna().nunique())),
    )
    .reset_index()
)

print(f"Exploded event-district records: {len(event_districts):,}")
print(f"Event parse audit records: {len(event_parse_audit):,}")
display(event_counts_by_name.head())

Exploded event-district records: 24,628
Event parse audit records: 59


,district_review_key,flood_events,inventory_district_names,lgd_codes_seen,lgd_code_count
0,,51,,603,1
1,& parts of himachal pradesh,2,& Parts of Himachal Pradesh,,0
2,& parts of uttar pradesh,1,& Parts of Uttar Pradesh,,0
3,& various parts of nagaland,1,& various parts of Nagaland,,0
4,11 districts,1,11 districts,,0


## Rebuild Audited One-to-One Join

Repeated district names and unmatched keys are still audited. The numeric debugging table uses only one-to-one district-name keys across the district summary tables and `DFSI.csv`.

In [6]:
impact = flood_impact.reset_index(names="impact_source_row_id").copy()
impact["district_review_key"] = impact["Dist_Name"].map(make_review_key)

area = flooded_area.reset_index(names="area_source_row_id").copy()
area["district_review_key"] = area["Dist_Name"].map(make_review_key)

published = dfsi.reset_index(names="dfsi_source_row_id").rename(columns={"Unnamed: 0": "DFSI_Dist_Name"}).copy()
published["district_review_key"] = published["DFSI_Dist_Name"].map(make_review_key)
published["state_review_key"] = published["State_Name"].map(make_review_key)

impact_name_counts = impact["district_review_key"].value_counts(dropna=False).rename("impact_name_count")
area_name_counts = area["district_review_key"].value_counts(dropna=False).rename("area_name_count")
published_name_counts = published["district_review_key"].value_counts(dropna=False).rename("published_name_count")

all_keys = pd.Index([])
for key_series in [impact["district_review_key"], area["district_review_key"], published["district_review_key"], event_counts_by_name["district_review_key"]]:
    all_keys = all_keys.union(pd.Index(key_series.dropna().unique()))

key_audit = pd.DataFrame({"district_review_key": all_keys})
key_audit = key_audit.merge(impact_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(area_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(published_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(event_counts_by_name[["district_review_key", "flood_events", "lgd_code_count", "lgd_codes_seen"]], on="district_review_key", how="left")

for column in ["impact_name_count", "area_name_count", "published_name_count", "flood_events", "lgd_code_count"]:
    key_audit[column] = key_audit[column].fillna(0).astype(int)

unmatched_records = key_audit.query(
    "impact_name_count == 0 or area_name_count == 0 or published_name_count == 0 or flood_events == 0"
).copy()

multiply_matched_records = key_audit.query(
    "impact_name_count > 1 or area_name_count > 1 or published_name_count > 1 or lgd_code_count > 1"
).copy()

one_to_one_keys = key_audit.query(
    "impact_name_count == 1 and area_name_count == 1 and published_name_count == 1"
)["district_review_key"]

candidate = (
    impact[impact["district_review_key"].isin(one_to_one_keys)]
    .merge(
        area[area["district_review_key"].isin(one_to_one_keys)],
        on="district_review_key",
        how="inner",
        suffixes=("_impact", "_area"),
        validate="one_to_one",
    )
    .merge(
        event_counts_by_name[["district_review_key", "flood_events", "inventory_district_names", "lgd_codes_seen", "lgd_code_count"]],
        on="district_review_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        published[published["district_review_key"].isin(one_to_one_keys)][["dfsi_source_row_id", "DFSI_Dist_Name", "State_Name", "state_review_key", "DFSI", "district_review_key"]],
        on="district_review_key",
        how="inner",
        validate="one_to_one",
    )
)

candidate["flood_events_missing_from_inventory"] = candidate["flood_events"].isna()
candidate["flood_events"] = candidate["flood_events"].fillna(0)

print(f"One-to-one candidate rows: {len(candidate):,}")
print(f"Unmatched audit rows: {len(unmatched_records):,}")
print(f"Multiply matched audit rows: {len(multiply_matched_records):,}")

One-to-one candidate rows: 720
Unmatched audit rows: 323
Multiply matched audit rows: 6


## Explicit Numeric Conversion and Component Audit

Each component is converted with `pd.to_numeric(..., errors="coerce").astype("float64")`.

In [7]:
comparison = candidate.copy()

comparison["component_flood_events"] = pd.to_numeric(comparison["flood_events"], errors="coerce").astype("float64")
comparison["component_mean_flood_duration"] = pd.to_numeric(comparison["Mean_Flood_Duration"], errors="coerce").astype("float64")
comparison["component_fatalities"] = pd.to_numeric(comparison["Human_fatality"], errors="coerce").astype("float64")
comparison["component_injured"] = pd.to_numeric(comparison["Human_injured"], errors="coerce").astype("float64")
comparison["component_flooded_area"] = pd.to_numeric(comparison["Corrected_Percent_Flooded_Area"], errors="coerce").astype("float64")
comparison["component_population"] = pd.to_numeric(comparison["Population"], errors="coerce").astype("float64")

component_columns = [
    "component_flood_events",
    "component_mean_flood_duration",
    "component_fatalities",
    "component_injured",
    "component_flooded_area",
    "component_population",
]

component_labels = {
    "component_flood_events": "flood_events",
    "component_mean_flood_duration": "mean_flood_duration",
    "component_fatalities": "fatalities",
    "component_injured": "injured",
    "component_flooded_area": "flooded_area",
    "component_population": "population",
}

source_column_lookup = {
    "component_flood_events": "count distinct UEI after exploding India_Flood_Inventory_v3.Districts",
    "component_mean_flood_duration": "District_FloodImpact.Mean_Flood_Duration",
    "component_fatalities": "District_FloodImpact.Human_fatality",
    "component_injured": "District_FloodImpact.Human_injured",
    "component_flooded_area": "District_FloodedArea.Corrected_Percent_Flooded_Area",
    "component_population": "District_FloodImpact.Population",
}

component_numeric_audit = pd.DataFrame([
    {
        "component": component_labels[column],
        "working_column": column,
        "exact_source_column_used": source_column_lookup[column],
        "dtype_after_conversion": str(comparison[column].dtype),
        "row_count": len(comparison),
        "minimum": comparison[column].min(skipna=True),
        "maximum": comparison[column].max(skipna=True),
        "missing_count": int(comparison[column].isna().sum()),
        "zero_count": int((comparison[column] == 0).sum()),
        "negative_count": int((comparison[column] < 0).sum()),
    }
    for column in component_columns
])

display(component_numeric_audit)
component_numeric_audit.to_csv(TABLE_DIR / "dfsi_component_numeric_audit.csv", index=False)

,component,working_column,exact_source_column_used,dtype_after_conversion,row_count,minimum,maximum,missing_count,zero_count,negative_count
0,flood_events,component_flood_events,count distinct UEI after exploding India_Flood_Inventory_v3.Districts,float64,720,0.000000,2.310000e+02,0,86,0
1,mean_flood_duration,component_mean_flood_duration,District_FloodImpact.Mean_Flood_Duration,float64,720,1.000000,3.500000e+01,11,0,0
2,fatalities,component_fatalities,District_FloodImpact.Human_fatality,float64,720,0.000000,1.726000e+03,0,36,0
3,injured,component_injured,District_FloodImpact.Human_injured,float64,720,0.000000,6.710000e+02,0,224,0
4,flooded_area,component_flooded_area,District_FloodedArea.Corrected_Percent_Flooded_Area,float64,720,0.000073,2.362193e+01,0,0,0
5,population,component_population,District_FloodImpact.Population,float64,720,7110.000000,1.340400e+07,0,0,0


## Negative and Missing Component Rows

Every row containing a negative component is displayed. The first 20 missing-component rows are displayed, while all missing rows are saved.

In [8]:
row_context_columns = [
    "district_review_key",
    "Dist_Name_impact",
    "DFSI_Dist_Name",
    "State_Name",
    *component_columns,
    "DFSI",
    "flood_events_missing_from_inventory",
    "lgd_codes_seen",
    "lgd_code_count",
]

negative_component_mask = comparison[component_columns].lt(0).any(axis=1)
negative_component_rows = comparison.loc[negative_component_mask, row_context_columns].rename(columns={"DFSI": "dfsi_published"})
display(negative_component_rows)
negative_component_rows.to_csv(TABLE_DIR / "dfsi_negative_component_rows.csv", index=False)

missing_component_mask = comparison[component_columns].isna().any(axis=1)
missing_component_rows = comparison.loc[missing_component_mask, row_context_columns].rename(columns={"DFSI": "dfsi_published"})
display(missing_component_rows.head(20))
missing_component_rows.to_csv(TABLE_DIR / "dfsi_missing_component_rows.csv", index=False)

print(f"Negative-component rows: {len(negative_component_rows):,}")
print(f"Missing-component rows preserved as missing: {len(missing_component_rows):,}")

,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,component_population,dfsi_published,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count


,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,component_population,dfsi_published,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
1,daman,Daman,Daman,DAMAN & DIU,0.0,NaN,0.0,0.0,1.544538,190125.0,5.233751,True,NaN,NaN
2,diu,Diu,Diu,DAMAN & DIU,0.0,NaN,0.0,0.0,0.819836,67412.0,4.269142,True,NaN,NaN
5,nicobars,Nicobars,Nicobars,ANDAMAN & NICOBAR,0.0,NaN,0.0,0.0,0.090612,27340.0,4.664727,True,NaN,NaN
118,dakshin bastar dantewada,Dakshin Bastar Dantewada,Dakshin Bastar Dantewada,CHHATTISGARH,2.0,NaN,0.0,0.0,1.673199,332104.0,7.263513,False,376,1.0
120,kondagaon,Kondagaon,Kondagaon,CHHATTISGARH,0.0,NaN,0.0,0.0,1.078483,656011.0,7.560308,True,NaN,NaN
121,gariaband,Gariaband,Gariaband,CHHATTISGARH,0.0,NaN,0.0,0.0,0.505736,777859.0,7.285327,True,NaN,NaN
133,surajpur,Surajpur,Surajpur,CHHATTISGARH,0.0,NaN,0.0,0.0,1.220060,917283.0,7.792234,True,NaN,NaN
142,east,East,East,DELHI,19.0,NaN,0.0,0.0,0.100000,1368632.0,NaN,False,505,1.0
309,lakshadweep,Lakshadweep,Lakshadweep,LAKSHADWEEP,0.0,NaN,0.0,0.0,4.000000,60324.0,4.858477,True,NaN,NaN
656,almora,Almora\n,Almora\n,UTTARAKHAND,20.0,NaN,0.0,0.0,0.648573,618943.0,7.102979,False,45,1.0


Negative-component rows: 0
Missing-component rows preserved as missing: 11


## Corrected Log-Space DFSI

The calculation is performed in log space. A float64 product/ratio is calculated only for verification; no integer product is constructed.

In [9]:
for column in component_columns:
    assert comparison[column].dropna().ge(0).all(), f"Negative values found in {column}"

complete_component_mask = comparison[component_columns].notna().all(axis=1)

comparison["dfsi_float64_product"] = (
    (1.0 + comparison["component_flood_events"])
    / (
        (1.0 + comparison["component_mean_flood_duration"])
        * (1.0 + comparison["component_fatalities"]) ** 2.0
        * (1.0 + comparison["component_injured"])
        * (1.0 + comparison["component_flooded_area"])
        * comparison["component_population"]
    )
)

assert comparison.loc[complete_component_mask, "dfsi_float64_product"].gt(0).all(), "Complete-row product must be positive"

comparison["dfsi_calculated"] = (
    np.log10(1.0 + comparison["component_flood_events"])
    - np.log10(1.0 + comparison["component_mean_flood_duration"])
    - 2.0 * np.log10(1.0 + comparison["component_fatalities"])
    - np.log10(1.0 + comparison["component_injured"])
    - np.log10(1.0 + comparison["component_flooded_area"])
    - np.log10(comparison["component_population"])
)

comparison["dfsi_calculated_from_product"] = np.log10(comparison["dfsi_float64_product"])
comparison["logspace_product_abs_diff"] = (comparison["dfsi_calculated"] - comparison["dfsi_calculated_from_product"]).abs()
comparison["absolute_difference"] = (comparison["dfsi_calculated"] - comparison["DFSI"]).abs()
comparison["rank_difference"] = comparison["dfsi_calculated"].rank(ascending=False) - comparison["DFSI"].rank(ascending=False)

comparison_columns = [
    "district_review_key",
    "Dist_Name_impact",
    "DFSI_Dist_Name",
    "State_Name",
    "component_flood_events",
    "component_mean_flood_duration",
    "component_fatalities",
    "component_injured",
    "component_flooded_area",
    "Percent_Flooded_Area",
    "Parmanent_Water",
    "component_population",
    "dfsi_float64_product",
    "dfsi_calculated",
    "dfsi_calculated_from_product",
    "logspace_product_abs_diff",
    "DFSI",
    "absolute_difference",
    "rank_difference",
    "flood_events_missing_from_inventory",
    "lgd_codes_seen",
    "lgd_code_count",
]

corrected_comparison = comparison[comparison_columns].rename(columns={"DFSI": "dfsi_published"})
display(corrected_comparison.head())
corrected_comparison.to_csv(TABLE_DIR / "dfsi_corrected_comparison.csv", index=False)

print("Maximum absolute difference between log-space and float64 product calculations:", corrected_comparison["logspace_product_abs_diff"].max())

,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,Percent_Flooded_Area,Parmanent_Water,component_population,dfsi_float64_product,dfsi_calculated,dfsi_calculated_from_product,logspace_product_abs_diff,dfsi_published,absolute_difference,rank_difference,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
0,dadra & nagar haveli,Dadra & Nagar Haveli,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,0.0,1.0,10.0,0.0,2.969168,3.614705,0.645537,618459.0,1.683349e-09,-8.773826,-8.773826,0.0,9.820211,18.594037,-442.0,True,NaN,NaN
1,daman,Daman,Daman,DAMAN & DIU,0.0,NaN,0.0,0.0,1.544538,2.853534,1.308996,190125.0,NaN,NaN,NaN,NaN,5.233751,NaN,NaN,True,NaN,NaN
2,diu,Diu,Diu,DAMAN & DIU,0.0,NaN,0.0,0.0,0.819836,5.799818,4.979982,67412.0,NaN,NaN,NaN,NaN,4.269142,NaN,NaN,True,NaN,NaN
3,north & middle andaman,North & Middle Andaman,North & Middle Andaman,ANDAMAN & NICOBAR,0.0,1.0,0.0,0.0,1.314817,1.314817,0.000000,101677.0,2.124372e-06,-5.672769,-5.672769,0.0,7.462575,13.135344,-693.0,True,NaN,NaN
4,south andaman,South Andaman,South Andaman,ANDAMAN & NICOBAR,0.0,2.0,1.0,0.0,1.443452,1.443452,0.000000,257900.0,1.322402e-07,-6.878636,-6.878636,0.0,8.360463,15.239100,-632.0,True,NaN,NaN


Maximum absolute difference between log-space and float64 product calculations: 1.7763568394002505e-15


## Corrected Metrics and Largest Discrepancies

In [10]:
finite_rows = corrected_comparison.replace([np.inf, -np.inf], np.nan).dropna(subset=["dfsi_calculated", "dfsi_published"])

pearson = finite_rows["dfsi_calculated"].corr(finite_rows["dfsi_published"], method="pearson") if len(finite_rows) >= 2 else np.nan
spearman = finite_rows["dfsi_calculated"].corr(finite_rows["dfsi_published"], method="spearman") if len(finite_rows) >= 2 else np.nan

corrected_metrics = pd.DataFrame([
    {"metric": "comparison_rows", "value": len(corrected_comparison)},
    {"metric": "complete_component_rows", "value": int(complete_component_mask.sum())},
    {"metric": "missing_component_rows_preserved", "value": int((~complete_component_mask).sum())},
    {"metric": "finite_calculated_dfsi_rows", "value": len(finite_rows)},
    {"metric": "negative_component_rows", "value": int(negative_component_mask.sum())},
    {"metric": "complete_row_product_positive", "value": bool(corrected_comparison.loc[complete_component_mask, "dfsi_float64_product"].gt(0).all())},
    {"metric": "max_logspace_vs_product_abs_diff", "value": corrected_comparison["logspace_product_abs_diff"].max()},
    {"metric": "pearson_correlation", "value": pearson},
    {"metric": "spearman_rank_correlation", "value": spearman},
    {"metric": "mean_absolute_difference", "value": finite_rows["absolute_difference"].mean() if len(finite_rows) else np.nan},
    {"metric": "median_absolute_difference", "value": finite_rows["absolute_difference"].median() if len(finite_rows) else np.nan},
    {"metric": "maximum_absolute_difference", "value": finite_rows["absolute_difference"].max() if len(finite_rows) else np.nan},
    {"metric": "flood_event_unmatched_rows_treated_as_zero_events", "value": int(corrected_comparison["flood_events_missing_from_inventory"].sum())},
])

display(corrected_metrics)
corrected_metrics.to_csv(TABLE_DIR / "dfsi_corrected_reproduction_metrics.csv", index=False)

largest_discrepancies = (
    corrected_comparison.dropna(subset=["absolute_difference"])
    .sort_values("absolute_difference", ascending=False)
    .head(30)
)

display(largest_discrepancies)
largest_discrepancies.to_csv(TABLE_DIR / "dfsi_corrected_largest_discrepancies.csv", index=False)

,metric,value
0,comparison_rows,720
1,complete_component_rows,709
2,missing_component_rows_preserved,11
3,finite_calculated_dfsi_rows,709
4,negative_component_rows,0
5,complete_row_product_positive,True
6,max_logspace_vs_product_abs_diff,0.0
7,pearson_correlation,-0.934496
8,spearman_rank_correlation,-0.934474
9,mean_absolute_difference,23.636519


,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,Percent_Flooded_Area,Parmanent_Water,component_population,dfsi_float64_product,dfsi_calculated,dfsi_calculated_from_product,logspace_product_abs_diff,dfsi_published,absolute_difference,rank_difference,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
671,north twenty four pargan*,North Twenty Four Pargan*,North Twenty Four Pargan*,WEST BENGAL,0.0,19.0,294.0,16.0,11.225591,18.664847,7.439256,10544399.0,2.621714e-16,-15.581415,-15.581415,0.000000e+00,18.770538,34.351952,704.0,True,NaN,NaN
104,patna,Patna,Patna,BIHAR,71.0,16.0,381.0,30.0,18.820619,20.269812,1.449193,6868950.0,6.876819e-15,-14.162612,-14.162612,0.000000e+00,19.300564,33.463177,701.0,False,520,1.0
78,purba champaran,Purba Champaran,Purba Champaran,BIHAR,0.0,19.0,184.0,20.0,9.387441,9.508604,0.121163,6322257.0,1.059318e-15,-14.974973,-14.974973,0.000000e+00,18.295788,33.270761,697.0,True,NaN,NaN
661,darjiling,Darjiling,Darjiling,WEST BENGAL,0.0,11.0,548.0,63.0,1.918361,2.014600,0.096239,1849559.0,8.003627e-16,-15.096713,-15.096713,1.776357e-15,18.018981,33.115695,686.0,True,NaN,NaN
665,murshidabad,Murshidabad,Murshidabad,WEST BENGAL,44.0,23.0,435.0,9.0,10.404534,11.638689,1.234154,8350690.0,1.035687e-14,-13.984772,-13.984772,1.776357e-15,18.910884,32.895655,699.0,False,319,1.0
159,ahmadabad,Ahmadabad,Ahmadabad,GUJARAT,0.0,12.0,177.0,23.0,4.099835,4.222293,0.122458,8738876.0,2.269832e-15,-14.644006,-14.644006,1.776357e-15,18.117122,32.761129,688.0,True,NaN,NaN
10,guntur,Guntur,Guntur,ANDHRA PRADESH,33.0,8.0,188.0,108.0,9.242779,10.242307,0.999528,5374652.0,1.762455e-14,-13.753882,-13.753882,1.776357e-15,18.832069,32.585951,694.0,False,506,1.0
673,medinipur west,Medinipur West,Medinipur West,WEST BENGAL,0.0,10.0,294.0,10.0,7.812548,7.838526,0.025978,5510554.0,1.955572e-15,-14.708726,-14.708726,0.000000e+00,17.869046,32.577772,679.0,True,NaN,NaN
389,thane,Thane,Thane,MAHARASHTRA,97.0,3.0,418.0,204.0,1.253628,3.515191,2.261563,9950177.0,3.035786e-14,-13.517729,-13.517729,1.776357e-15,18.861789,32.379518,692.0,False,497,1.0
664,maldah,Maldah,Maldah,WEST BENGAL,17.0,11.0,221.0,53.0,9.357036,11.776244,2.419208,4260890.0,1.277191e-14,-13.893744,-13.893744,0.000000e+00,18.258130,32.151874,686.0,False,,0.0


## Answers

In [11]:
answer_summary = pd.DataFrame([
    {
        "question": "A. Was the non-positive argument problem caused by numeric overflow?",
        "answer": "No. The prior problem came from implementing the equation as raw subtraction inside log10. The corrected log-space form and the float64 product verification produce positive products for every complete row.",
    },
    {
        "question": "B. Were any incorrect columns mapped?",
        "answer": "No incorrect numeric source columns were identified in this debugging pass. The remaining uncertainty is identifier-level: the district summary files lack LGD codes and some district names are ambiguous. Flooded area still uses Corrected_Percent_Flooded_Area as the most defensible mapped column, with the uncorrected area retained for review.",
    },
    {
        "question": "C. How many rows now have finite calculated DFSI?",
        "answer": str(int(corrected_metrics.loc[corrected_metrics["metric"] == "finite_calculated_dfsi_rows", "value"].iloc[0])),
    },
    {
        "question": "D. What is the corrected Spearman rank correlation?",
        "answer": str(corrected_metrics.loc[corrected_metrics["metric"] == "spearman_rank_correlation", "value"].iloc[0]),
    },
    {
        "question": "E. What remains necessary for exact reproduction?",
        "answer": "A fully documented identifier crosswalk, preferably LGD district/state codes for DFSI.csv, District_FloodImpact.csv, and District_FloodedArea.csv; resolution of repeated district names; confirmation of the flooded-area variant and any scaling/normalisation used by the publication; and treatment rules for missing Mean_Flood_Duration rows.",
    },
])

display(answer_summary)

,question,answer
0,A. Was the non-positive argument problem caused by numeric overflow?,No. The prior problem came from implementing the equation as raw subtraction inside log10. The corrected log-space form and the float64 product verification produce positive pr...
1,B. Were any incorrect columns mapped?,No incorrect numeric source columns were identified in this debugging pass. The remaining uncertainty is identifier-level: the district summary files lack LGD codes and some di...
2,C. How many rows now have finite calculated DFSI?,709
3,D. What is the corrected Spearman rank correlation?,-0.9344741971467349
4,E. What remains necessary for exact reproduction?,"A fully documented identifier crosswalk, preferably LGD district/state codes for DFSI.csv, District_FloodImpact.csv, and District_FloodedArea.csv; resolution of repeated distri..."
